# FormReader - TÜBİTAK 3501 Projesi

## Hücre Rehberi
- **Hücre 1-4**: Form işleme (opsiyonel)
- **Hücre 5**: Sentetik veri üretimi (50K) - ~30-40 dk
- **Hücre 6**: Model eğitimi (Deney #003 Bug Fix) - ~7-8 saat
- **Hücre 7**: Sonuç testi (DENEY değişkenini değiştirerek farklı deneyleri test et)

---
**GECE İÇİN**: Sadece Hücre 6'yı çalıştır! (veri zaten mevcut)

**VERSİYONLAMA**: Her deney ayrı klasöre kaydedilir (`deney002/`, `deney003/`, ...)

In [ ]:
# ============================================================
# HÜCRE 1: Kütüphaneleri Yükle
# ============================================================
import sys
import os
import cv2
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('../src')) 

from preprocessing import find_boxes
from utils import parse_xml_labels, match_and_crop
from ocr_engine import OCREngine

def show(img, title="Resim"):
    plt.figure(figsize=(10,10))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()

print("[OK] Kütüphaneler yüklendi")

In [ ]:
# ============================================================
# HÜCRE 2: Formu Oku ve Kutuları Bul (OPSİYONEL)
# ============================================================
IMAGE_PATH = "../data/raw/S25C-925103023031.jpg"

processed_image, boxes = find_boxes(IMAGE_PATH)
print(f"Toplam {len(boxes)} adet kutu bulundu.")
show(processed_image, "Bulunan Kutular")

In [ ]:
# ============================================================
# HÜCRE 3: Etiketleri Eşleştir ve Kes (OPSİYONEL)
# ============================================================
XML_PATH = "../data/xml_labels/S25C-925103023031.xml"

xml_labels = parse_xml_labels(XML_PATH)
labeled_data = match_and_crop(processed_image, boxes, xml_labels, output_folder="../data/processed")

for item in labeled_data[:5]:
    print(f"Etiket: {item['label']}")
    show(item['roi_image'], item['label'])

In [ ]:
# ============================================================
# HÜCRE 4: Base Model ile Test (OPSİYONEL)
# ============================================================
ocr = OCREngine()

print("\n--- BASE MODEL OKUMA SONUÇLARI ---")
for item in labeled_data[:10]:
    text = ocr.predict(item['roi_image'])
    print(f"Kutu: {item['label']} -> Okunan: {text}")

In [ ]:
# ============================================================
# HÜCRE 5: SENTETİK VERİ ÜRETİMİ (200.000 ADET)
# ============================================================
# Tahmini süre: ~2 saat
# Disk: ~8-10 GB
# ============================================================

from data_generator import generate_dataset
import pandas as pd

FONT_DIR = "../data/fonts"
OUTPUT_DIR = "../data/synthetic"
COUNT = 200000  # Deney #005 için 200K

print("="*60)
print("SENTETİK VERİ ÜRETİMİ BAŞLIYOR")
print(f"Hedef: {COUNT:,} görüntü")
print("="*60)

csv_path = generate_dataset(FONT_DIR, OUTPUT_DIR, count=COUNT)

# Kontrol
df = pd.read_csv(csv_path)
print(f"\n[TAMAMLANDI] Toplam: {len(df):,} veri")
print(f"\nKategori dağılımı:")
print(df['category'].value_counts())

In [ ]:
# ============================================================
# HÜCRE 6: MODEL EĞİTİMİ (Deney #006 - PaddleOCR)
# ============================================================
# Mimari: PP-OCRv4 Server Recognition (SVTR + CTC)
# Decoder: CTC (karakter bazlı - tokenizer sorunu YOK!)
# Türkçe dict.txt ile ş,ğ,ü,ı,ö,ç tek sınıf olarak tanımlı
# GPU: RTX 4070 Super 12GB
# ============================================================

import importlib
import sys
sys.path.insert(0, '../src')

import paddle_trainer
importlib.reload(paddle_trainer)
from paddle_trainer import train

EPOCHS = 100
BATCH_SIZE = 32

print("="*60)
print("DENEY #006: PaddleOCR Fine-Tuning")
print("Mimari: PP-OCRv4 Server (SVTR + CTC)")
print(f"Epochs: {EPOCHS} | Batch: {BATCH_SIZE}")
print("="*60)

train(epochs=EPOCHS, batch_size=BATCH_SIZE)

In [ ]:
# ============================================================
# HÜCRE 7: SONUÇ TESTİ (PaddleOCR)
# ============================================================

import importlib
import sys
sys.path.insert(0, '../src')

import paddle_test
importlib.reload(paddle_test)
from paddle_test import test_paddle_model

DENEY = "deney006"
model_path = f"../models/paddle-turkish-handwritten/{DENEY}"

print(f"Test edilen deney: {DENEY}")
print(f"Model yolu: {model_path}")
print()

test_paddle_model(model_dir=model_path)